# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking**

My lane is Content Opportunity Scoring — scoring each page by its decline risk to build a priority review queue. The underlying model is a binary classifier that predicts whether a page is declining or not. Its output probability (between 0 and 1) then becomes the score used to rank all pages, so editors know exactly which ones to review first.

- It is **not pure clustering** — we already have a meaningful label and are not discovering unknown groups.
- It is **not regression** — we are not predicting how much traffic will drop, only whether the page is at risk.
- The task starts as **binary classification** (declining or not) and its probability output drives a **ranked scoring queue** — which is the real product editors use.

In short: classification is the method, scoring/ranking is the output and the action.

In [ ]:
# Section 1 — confirm task type
print("Lane      : Refresh / Content Opportunity Scoring")
print("Task type : Scoring / Ranking")
print("Method    : Binary classifier -> probability score -> ranked review queue")
print("Output    : Each page gets a score 0-100; top-K pages go to editorial review")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target column:** `is_declining_label`

**How it is created:**
```
trend_pct       = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d x 100
trend_direction = 'down'  when trend_pct < -20%
is_declining_label = 1    when trend_direction == 'down', else 0
```

**Where it comes from — rule-derived, not a direct observation:**
The label is produced by a threshold rule on impression change, not by a human editor or a confirmed business outcome. This means:
- The label captures a real, measured traffic decline — that part is data-driven.
- The -20% threshold is a human design choice; it is a proxy for the true goal (content that needs updating).
- It cannot tell us whether updating the page will recover traffic — that requires a separate causal study.

**Critical leakage warning:**
`trend_direction` and `trend_pct` are the direct sources of this label. They must **never** be used as model features — using them would mean the model just learns the rule, not the real pattern.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create the target label the same way the pipeline does
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

total = len(df)
declining = df['is_declining_label'].sum()

print("Target column: is_declining_label")
print(f"  Label = 1 (declining)    : {declining:,}  ({declining/total:.1%})")
print(f"  Label = 0 (not declining): {total - declining:,}  ({(total-declining)/total:.1%})")
print()
print("NEVER use as features: trend_direction, trend_pct (they define the label)")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

This directly maps to the real editorial action: of the top 50 pages the model flags for review, how many actually need it? This matches how the output is used — an editor reviews the top list, so accuracy on that list is what matters.

- Baseline hand rule: Precision@50 = **0.240** (only 12 of 50 picks are correct)
- Random forest model: Precision@50 = **0.740** (37 of 50 picks are correct)
- That is roughly a **3x improvement** over the fixed rule

**Secondary metric: ROC-AUC**
Measures how well the model separates declining pages from healthy ones across all thresholds. A score of 0.5 = random guessing. The random forest achieved 0.750 vs 0.627 for the hand rule.

**Why not accuracy?**
The label is nearly 50/50 (54.2% declining). Accuracy would be misleading — a model that flags everything gets 54% accuracy while being useless to editors.

**What 'good' means before training:**
Any model that scores below Precision@50 = 0.240 on the test set has failed to improve on a simple if-statement.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Baseline hand rule score (from the pipeline)
df['baseline_score'] = df['impressions_90d'] * (1 - df['ctr'].fillna(0) / 100)
top50 = df.nlargest(50, 'baseline_score')['is_declining_label']
precision_baseline = top50.mean()

print("Success metric: Precision@50")
print("-" * 40)
print(f"Base rate (full data):        {df['is_declining_label'].mean():.1%}")
print(f"Hand rule Precision@50:       {precision_baseline:.3f}  ({precision_baseline:.1%})")
print(f"Random forest Precision@50:   0.740  (74.0%)")
print(f"Improvement:                  ~3x over the hand rule")
print()
print("Target: beat 0.240 on the held-out client test set.")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page, measured over a 90-day trailing window.**

Each row represents a single webpage from a single client with all its search and engagement signals aggregated over 90 days. The model scores each page independently.

- Grain: `content_id` — unique per page (pseudonymised)
- 30,000 pages across 32 clients
- IDs (`content_id`, `client_id`) are for grouping only — never model features
- `client_id` is used for train/test splits — no client appears in both train and test

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Unit of analysis: ONE ROW = ONE CONTENT PAGE")
print(f"Total pages  : {len(df):,}")
print(f"Clients      : {df['client_id'].nunique()}")
print(f"Unique pages : {df['content_id'].is_unique}")
print()

# Show the key columns that define the framing
cols = [
    'content_id',        # identifier — grouping only
    'content_type',      # what kind of page
    'content_age_days',  # how old
    'avg_position',      # where it ranks (0 = no data)
    'impressions_90d',   # search visibility
    'ctr',               # click rate (x100 percentage)
    'is_declining_label' # TARGET: what we predict
]

print("Sample — one row = one content page:")
print(df[cols].head(8).to_string(index=False))
print()
print("Target column distribution:")
print(df['is_declining_label'].value_counts().rename({0: 'Healthy (0)', 1: 'Declining (1)'}).to_string())

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**The hand rule already exists — and it only gets 24% right.**

The baseline flags pages with high impressions but low CTR. Reasonable, but 76% of its top-50 picks are false alarms. Here is why a single rule cannot do better:

**1. Many signals interact, not one.**
A page might have declining impressions AND a weak position AND thin content AND high age — all at once. No single threshold captures this combination. ML learns the joint pattern across all signals.

**2. Patterns differ by content type.**
A `feedly article` has completely different typical CTR and position values than a `keyword article`. A rule calibrated for one type misfires on the others. ML learns type-specific patterns from data.

**3. Patterns differ by client.**
32 clients have different sizes, strategies, and traffic baselines. A threshold that works for a large client is wrong for a small one. ML trained with a client-holdout split learns signals that generalise.

**4. We need a probability score, not a binary flag.**
A fixed rule gives yes/no. ML gives a probability (e.g. 0.87 vs 0.52), which lets us rank pages by urgency and focus editor time on the highest-risk ones first.

**What ML cannot do here (being honest):**
- It cannot prove that updating a page will recover traffic — that needs a causal experiment.
- It cannot flag pages with zero impressions — those are a discoverability problem, not decay.
- It cannot predict exactly how much traffic will drop — only whether the risk threshold is crossed.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show decline rate varies by content type — a single rule cannot cover all types
print("Decline rate by content type:")
print("-" * 40)
print(df.groupby('content_type')['is_declining_label'].mean().round(3).to_string())
print()

# Show decline rate varies by position tier — another dimension a rule cannot capture
print("Decline rate by position tier:")
print("-" * 40)
print(df.groupby('position_tier')['is_declining_label'].mean().round(3).to_string())
print()
print("Each group has a different decline rate.")
print("A single if-statement threshold would misfire on most combinations.")
print("ML learns these interactions from data — a fixed rule cannot.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.